# Notebook 4: Specialized RAG Techniques

**Duration:** 120 minutes  
**Level:** Advanced  

---

## Learning Objectives

By the end of this notebook, you will be able to:

- ✅ Implement Corrective RAG (CRAG) patterns
- ✅ Build GraphRAG with knowledge graphs
- ✅ Handle long-context documents effectively
- ✅ Implement multimodal RAG with text and images
- ✅ Choose appropriate techniques for specific use cases

---

## Table of Contents

1. [Introduction to Specialized RAG](#1-introduction-to-specialized-rag)
2. [Environment Setup](#2-environment-setup)
3. [Corrective RAG (CRAG)](#3-corrective-rag-crag)
4. [GraphRAG with Knowledge Graphs](#4-graphrag-with-knowledge-graphs)
5. [Long-Context RAG](#5-long-context-rag)
6. [Multimodal RAG (Text + Images)](#6-multimodal-rag)
7. [Comparison & Selection Guide](#7-comparison-selection-guide)
8. [Exercises & Challenges](#8-exercises-challenges)
9. [Summary & Next Steps](#9-summary-next-steps)

---

## 1. Introduction to Specialized RAG

### Why Specialized Techniques?

Standard RAG works well for many cases, but specialized techniques excel in specific scenarios:

| Technique | Best For | Key Benefit |
|-----------|----------|-------------|
| **CRAG** | High-stakes domains (medical, legal) | Validates and corrects retrieved info |
| **GraphRAG** | Complex relationships & multi-hop reasoning | Leverages structured knowledge |
| **Long-Context** | Full document understanding | Preserves document structure |
| **Multimodal** | Documents with images/diagrams | Processes visual information |

### Technique Overview

**Corrective RAG (CRAG):**
```
Query → Retrieve → Evaluate Quality → If Low: Web Search → Correct → Generate
```

**GraphRAG:**
```
Query → Extract Entities → Graph Traversal → Find Related Info → Generate
```

**Long-Context RAG:**
```
Large Doc → Hierarchical Chunking → Parent-Child Relationships → Selective Retrieval
```

**Multimodal RAG:**
```
Query → Retrieve Text + Images → Vision Model → Combine Modalities → Generate
```

## 2. Environment Setup

In [5]:
# Standard library
import os
import json
import warnings
from typing import List, Dict, Any, Optional, Tuple
import hashlib

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.utils import Secret

warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


In [6]:
# Load environment
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY required")

print("✅ Environment configured")

✅ Environment configured


## 3. Corrective RAG (CRAG)

CRAG adds quality assessment and correction to standard RAG.

### 3.1 Understanding CRAG

CRAG workflow:
1. Retrieve documents
2. **Evaluate** retrieval quality
3. If quality low → **Correct** (web search, rerank, filter)
4. Generate answer
5. **Validate** answer quality
6. If needed → **Regenerate**

In [7]:
# Sample multi-hop dataset (requires combining info from multiple docs)
def create_multihop_dataset() -> List[Document]:
    """
    Create dataset requiring multi-hop reasoning.
    """
    docs_data = [
        {
            "content": "The Eiffel Tower was designed by Gustave Eiffel and completed in 1889.",
            "meta": {"title": "Eiffel Tower Construction", "topic": "landmarks"}
        },
        {
            "content": "Gustave Eiffel was born in Dijon, France on December 15, 1832.",
            "meta": {"title": "Gustave Eiffel Biography", "topic": "people"}
        },
        {
            "content": "The Eiffel Tower stands 330 meters tall and is located in Paris.",
            "meta": {"title": "Eiffel Tower Specifications", "topic": "landmarks"}
        },
        {
            "content": "Dijon is the capital of the Burgundy region in eastern France.",
            "meta": {"title": "Dijon Geography", "topic": "geography"}
        },
        {
            "content": "The Statue of Liberty was a gift from France to the United States in 1886.",
            "meta": {"title": "Statue of Liberty", "topic": "landmarks"}
        }
    ]
    
    return [Document(content=d["content"], meta=d["meta"]) for d in docs_data]

multihop_docs = create_multihop_dataset()
multihop_store = InMemoryDocumentStore()
multihop_store.write_documents(multihop_docs)

print(f"✅ Created multi-hop dataset with {len(multihop_docs)} documents")

✅ Created multi-hop dataset with 5 documents


In [8]:
def evaluate_retrieval_quality(query: str, documents: List[Document]) -> Tuple[str, float, str]:
    """
    Evaluate quality of retrieved documents.
    
    Returns:
        (quality_level, confidence_score, reasoning)
        quality_level: 'high', 'medium', 'low'
    """
    if not documents:
        return 'low', 0.0, "No documents retrieved"
    
    # Simple heuristic: check document scores
    top_score = documents[0].score if hasattr(documents[0], 'score') else 1.0
    
    # Use LLM to assess relevance
    doc_summaries = "\n".join([f"- {doc.content[:100]}..." for doc in documents[:3]])
    
    prompt = f"""
    Assess whether these retrieved documents can answer the query well.
    
    Query: {query}
    
    Retrieved documents:
    {doc_summaries}
    
    Rate the retrieval quality as:
    - HIGH: Documents directly answer the query with complete information
    - MEDIUM: Documents partially answer but may need additional info
    - LOW: Documents are not relevant or insufficient
    
    Respond in format: QUALITY_LEVEL (confidence 0-1) - reasoning
    Example: HIGH (0.9) - Documents contain exact information needed
    """
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    response = llm.run(prompt=prompt)['replies'][0]
    
    # Parse response
    response_upper = response.upper()
    if 'HIGH' in response_upper:
        quality = 'high'
    elif 'LOW' in response_upper:
        quality = 'low'
    else:
        quality = 'medium'
    
    # Extract confidence if present
    try:
        confidence = float(response.split('(')[1].split(')')[0])
    except:
        confidence = 0.5
    
    return quality, confidence, response

# Test quality evaluation
retriever = InMemoryBM25Retriever(document_store=multihop_store)

test_queries = [
    "What is the height of the Eiffel Tower?",  # Should be HIGH
    "Where was the designer of the Eiffel Tower born?",  # Requires multi-hop
    "What is the population of Tokyo?",  # Not in docs - should be LOW
]

print("\nTesting Retrieval Quality Evaluation:\n")
for query in test_queries:
    result = retriever.run(query=query, top_k=3)
    docs = result['documents']
    
    quality, confidence, reasoning = evaluate_retrieval_quality(query, docs)
    
    print(f"Query: {query}")
    print(f"Quality: {quality.upper()} (confidence: {confidence:.2f})")
    print(f"Reasoning: {reasoning}\n")


Testing Retrieval Quality Evaluation:

Query: What is the height of the Eiffel Tower?
Quality: HIGH (confidence: 0.90)
Reasoning: HIGH (0.9) - The retrieved documents include precise details about the height of the Eiffel Tower (330 meters), directly answering the query with full information.

Query: Where was the designer of the Eiffel Tower born?
Quality: HIGH (confidence: 1.00)
Reasoning: HIGH (1.0) - The retrieved documents directly answer the query by providing the birthplace of Gustave Eiffel, the designer of the Eiffel Tower, stating that he was born in Dijon, France. This matches the required information from the query with complete precision.

Query: What is the population of Tokyo?
Quality: LOW (confidence: 1.00)
Reasoning: LOW (1.0) - The retrieved documents are not relevant to the query about the population of Tokyo. None of the documents contain any information about Tokyo or its population.



In [9]:
def corrective_rag(query: str, quality_threshold: float = 0.6) -> Dict[str, Any]:
    """
    Implement CRAG: assess quality and apply corrections.
    """
    print(f"\n{'='*80}")
    print(f"CRAG Processing: {query}")
    print(f"{'='*80}\n")
    
    # Step 1: Initial retrieval
    print("1️⃣  Initial retrieval...")
    retriever = InMemoryBM25Retriever(document_store=multihop_store)
    result = retriever.run(query=query, top_k=5)
    docs = result['documents']
    
    # Step 2: Evaluate quality
    print("2️⃣  Evaluating retrieval quality...")
    quality, confidence, reasoning = evaluate_retrieval_quality(query, docs)
    print(f"   Quality: {quality.upper()} (confidence: {confidence:.2f})")
    print(f"   {reasoning}")
    
    # Step 3: Apply correction if needed
    correction_applied = False
    if quality == 'low' or confidence < quality_threshold:
        print("\n3️⃣  ⚠️  Quality insufficient - applying corrections...")
        
        # Correction strategy 1: Retrieve more documents
        print("   Strategy: Retrieving more documents (top_k=10)")
        expanded_result = retriever.run(query=query, top_k=10)
        docs = expanded_result['documents']
        
        # Correction strategy 2: In production, would also try:
        # - Web search
        # - Query reformulation
        # - Different retrieval method
        
        correction_applied = True
    else:
        print("\n3️⃣  ✅ Quality sufficient - proceeding with retrieved docs")
    
    # Step 4: Generate answer
    print("\n4️⃣  Generating answer...")
    template = """
    Answer the question based on the provided information. Be precise and cite sources.
    
    Information:
    {% for doc in documents %}
    {{ doc.content }}
    {% endfor %}
    
    Question: {{ query }}
    
    If the information is insufficient, clearly state that.
    Answer:
    """
    
    prompt_builder = PromptBuilder(template=template)
    prompt = prompt_builder.run(documents=docs, query=query)['prompt']
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    answer = llm.run(prompt=prompt)['replies'][0]
    
    print("\n5️⃣  Answer generated\n")
    
    return {
        'answer': answer,
        'quality': quality,
        'confidence': confidence,
        'correction_applied': correction_applied,
        'docs_used': len(docs)
    }

# Test CRAG
for query in test_queries:
    result = corrective_rag(query)
    print(f"Answer: {result['answer']}")
    print(f"\nMetadata:")
    print(f"  Quality: {result['quality']}")
    print(f"  Correction applied: {result['correction_applied']}")
    print(f"  Documents used: {result['docs_used']}")


CRAG Processing: What is the height of the Eiffel Tower?

1️⃣  Initial retrieval...
2️⃣  Evaluating retrieval quality...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


   Quality: HIGH (confidence: 0.90)
   HIGH (0.9) - The first retrieved document directly provides the exact height of the Eiffel Tower, answering the query completely. The other documents add some context but are not necessary to answer the height question.

3️⃣  ✅ Quality sufficient - proceeding with retrieved docs

4️⃣  Generating answer...

5️⃣  Answer generated

Answer: The height of the Eiffel Tower is 330 meters.

Metadata:
  Quality: high
  Correction applied: False
  Documents used: 5

CRAG Processing: Where was the designer of the Eiffel Tower born?

1️⃣  Initial retrieval...
2️⃣  Evaluating retrieval quality...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


   Quality: HIGH (confidence: 0.90)
   HIGH (0.9) - The retrieved documents directly answer the query by providing the birthplace of the designer of the Eiffel Tower, Gustave Eiffel, specifically mentioning that he was born in Dijon, France.

3️⃣  ✅ Quality sufficient - proceeding with retrieved docs

4️⃣  Generating answer...

5️⃣  Answer generated

Answer: The designer of the Eiffel Tower, Gustave Eiffel, was born in Dijon, France.

Metadata:
  Quality: high
  Correction applied: False
  Documents used: 5

CRAG Processing: What is the population of Tokyo?

1️⃣  Initial retrieval...
2️⃣  Evaluating retrieval quality...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


   Quality: LOW (confidence: 1.00)
   LOW (1.0) - The retrieved documents are not relevant to the query about the population of Tokyo and do not contain any information related to Tokyo at all. They instead provide information about locations and landmarks in France and the United States.

3️⃣  ⚠️  Quality insufficient - applying corrections...
   Strategy: Retrieving more documents (top_k=10)

4️⃣  Generating answer...

5️⃣  Answer generated

Answer: The provided information does not contain details about the population of Tokyo. To find the population of Tokyo, additional sources would be needed.

Metadata:
  Quality: low
  Correction applied: True
  Documents used: 5


## 4. GraphRAG with Knowledge Graphs

GraphRAG leverages knowledge graphs for structured information.

### 4.1 Understanding GraphRAG

**Why Knowledge Graphs?**
- Explicit relationships between entities
- Multi-hop reasoning made easy
- Structured, queryable knowledge

**Note:** For production GraphRAG, you'd use Neo4j or similar. Here we'll simulate the concept.

In [10]:
# Simulate a knowledge graph (in production, use Neo4j)
class SimpleKnowledgeGraph:
    """
    Simplified knowledge graph for demonstration.
    In production, use Neo4j with proper Cypher queries.
    """
    
    def __init__(self):
        self.entities = {}
        self.relationships = []
    
    def add_entity(self, entity_id: str, entity_type: str, properties: Dict[str, Any]):
        """Add an entity to the graph."""
        self.entities[entity_id] = {
            'type': entity_type,
            'properties': properties
        }
    
    def add_relationship(self, from_id: str, relation: str, to_id: str, properties: Dict[str, Any] = None):
        """Add a relationship between entities."""
        self.relationships.append({
            'from': from_id,
            'relation': relation,
            'to': to_id,
            'properties': properties or {}
        })
    
    def find_entity(self, entity_id: str) -> Optional[Dict]:
        """Find entity by ID."""
        return self.entities.get(entity_id)
    
    def find_related(self, entity_id: str, relation: Optional[str] = None) -> List[Dict]:
        """Find entities related to given entity."""
        related = []
        for rel in self.relationships:
            if rel['from'] == entity_id:
                if relation is None or rel['relation'] == relation:
                    related.append({
                        'entity_id': rel['to'],
                        'relation': rel['relation'],
                        'entity': self.entities.get(rel['to'])
                    })
        return related
    
    def traverse(self, start_id: str, max_hops: int = 2) -> List[Dict]:
        """Traverse graph from starting entity."""
        visited = set()
        results = []
        
        def _traverse(entity_id: str, depth: int):
            if depth > max_hops or entity_id in visited:
                return
            
            visited.add(entity_id)
            entity = self.find_entity(entity_id)
            if entity:
                results.append({'id': entity_id, **entity, 'depth': depth})
            
            # Traverse neighbors
            related = self.find_related(entity_id)
            for r in related:
                _traverse(r['entity_id'], depth + 1)
        
        _traverse(start_id, 0)
        return results

# Create sample knowledge graph
kg = SimpleKnowledgeGraph()

# Add entities
kg.add_entity('eiffel_tower', 'Landmark', {
    'name': 'Eiffel Tower',
    'height': '330 meters',
    'location': 'Paris'
})
kg.add_entity('gustave_eiffel', 'Person', {
    'name': 'Gustave Eiffel',
    'birth_year': 1832,
    'birthplace': 'Dijon'
})
kg.add_entity('paris', 'City', {
    'name': 'Paris',
    'country': 'France'
})
kg.add_entity('dijon', 'City', {
    'name': 'Dijon',
    'region': 'Burgundy',
    'country': 'France'
})

# Add relationships
kg.add_relationship('eiffel_tower', 'DESIGNED_BY', 'gustave_eiffel')
kg.add_relationship('eiffel_tower', 'LOCATED_IN', 'paris')
kg.add_relationship('gustave_eiffel', 'BORN_IN', 'dijon')
kg.add_relationship('paris', 'PART_OF', 'france')
kg.add_relationship('dijon', 'PART_OF', 'france')

print("✅ Knowledge graph created with:")
print(f"   {len(kg.entities)} entities")
print(f"   {len(kg.relationships)} relationships")

✅ Knowledge graph created with:
   4 entities
   5 relationships


In [12]:
def extract_entities_from_query(query: str) -> List[str]:
    """
    Extract entity mentions from query using LLM.
    """
    prompt = f"""
    Extract the main entities (people, places, things) mentioned in this query.
    
    Query: {query}
    
    List only the entity names, one per line, nothing else.
    """
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    
    response = llm.run(prompt=prompt)['replies'][0]
    entities = [e.strip() for e in response.split('\n') if e.strip()]
    return entities

def graphrag_query(query: str, kg: SimpleKnowledgeGraph) -> Dict[str, Any]:
    """
    Answer query using knowledge graph traversal.
    """
    print(f"\n{'='*80}")
    print(f"GraphRAG Query: {query}")
    print(f"{'='*80}\n")
    
    # Step 1: Extract entities
    print("1️⃣  Extracting entities from query...")
    query_entities = extract_entities_from_query(query)
    print(f"   Found: {query_entities}")
    
    # Step 2: Find entities in graph
    print("\n2️⃣  Finding entities in knowledge graph...")
    matched_entities = []
    for entity_id, entity_data in kg.entities.items():
        entity_name = entity_data['properties'].get('name', '').lower()
        for query_entity in query_entities:
            if query_entity.lower() in entity_name or entity_name in query_entity.lower():
                matched_entities.append(entity_id)
                print(f"   ✅ Matched: {entity_id}")
                break
    
    if not matched_entities:
        print("   ⚠️  No entities found in knowledge graph")
        return {'answer': "I couldn't find relevant information in the knowledge graph.", 'path': []}
    
    # Step 3: Traverse graph
    print("\n3️⃣  Traversing knowledge graph...")
    all_results = []
    for entity_id in matched_entities:
        results = kg.traverse(entity_id, max_hops=2)
        all_results.extend(results)
        print(f"   Found {len(results)} connected entities from {entity_id}")
    
    # Step 4: Build context from graph
    print("\n4️⃣  Building context from graph...")
    context_parts = []
    for result in all_results:
        props = result['properties']
        context = f"{result['id']} ({result['type']}): " + ", ".join(
            f"{k}={v}" for k, v in props.items()
        )
        context_parts.append(context)
    
    context_str = "\n".join(context_parts)
    
    # Step 5: Generate answer
    print("\n5️⃣  Generating answer from graph context...\n")
    template = """
    Answer the question using the knowledge graph information.
    
    Knowledge Graph Info:
    {{ context }}
    
    Question: {{ query }}
    
    Answer:
    """
    
    prompt_builder = PromptBuilder(template=template)
    prompt = prompt_builder.run(context=context_str, query=query)['prompt']
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    answer = llm.run(prompt=prompt)['replies'][0]
    
    return {
        'answer': answer,
        'entities_matched': matched_entities,
        'graph_nodes_used': len(all_results),
        'context': context_str
    }

# Test GraphRAG
graph_queries = [
    "Where was the designer of the Eiffel Tower born?",  # Multi-hop
    "What city is the Eiffel Tower in?",  # Single hop
]

for query in graph_queries:
    result = graphrag_query(query, kg)
    print(f"Answer: {result['answer']}")
    print(f"\nMetadata:")
    print(f"  Entities matched: {result['entities_matched']}")
    print(f"  Graph nodes used: {result['graph_nodes_used']}")


GraphRAG Query: Where was the designer of the Eiffel Tower born?

1️⃣  Extracting entities from query...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


   Found: ['Eiffel Tower']

2️⃣  Finding entities in knowledge graph...
   ✅ Matched: eiffel_tower

3️⃣  Traversing knowledge graph...
   Found 4 connected entities from eiffel_tower

4️⃣  Building context from graph...

5️⃣  Generating answer from graph context...

Answer: The designer of the Eiffel Tower, Gustave Eiffel, was born in Dijon, France.

Metadata:
  Entities matched: ['eiffel_tower']
  Graph nodes used: 4

GraphRAG Query: What city is the Eiffel Tower in?

1️⃣  Extracting entities from query...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


   Found: ['Eiffel Tower']

2️⃣  Finding entities in knowledge graph...
   ✅ Matched: eiffel_tower

3️⃣  Traversing knowledge graph...
   Found 4 connected entities from eiffel_tower

4️⃣  Building context from graph...

5️⃣  Generating answer from graph context...

Answer: The Eiffel Tower is in Paris.

Metadata:
  Entities matched: ['eiffel_tower']
  Graph nodes used: 4


### 4.2 Production GraphRAG with Neo4j

For production, you'd use Neo4j:

```python
from neo4j import GraphDatabase

# Connect to Neo4j
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD"))
)

# Query with Cypher
def query_graph(query):
    with driver.session() as session:
        result = session.run(
            "MATCH (n:Entity)-[r]->(m:Entity) "
            "WHERE n.name CONTAINS $name "
            "RETURN n, r, m",
            name=entity_name
        )
        return list(result)
```

## 5. Long-Context RAG

Handle documents that exceed typical chunk sizes.

### 5.1 Hierarchical Chunking

In [13]:
def hierarchical_chunk_document(content: str, chunk_size: int = 500) -> List[Dict[str, Any]]:
    """
    Create hierarchical chunks with parent-child relationships.
    
    Returns list of chunks with metadata about hierarchy.
    """
    # Split into paragraphs (parent chunks)
    paragraphs = content.split('\n\n')
    
    chunks = []
    parent_id = 0
    
    for para in paragraphs:
        if not para.strip():
            continue
        
        parent_id += 1
        
        # If paragraph is small enough, use as-is
        if len(para) <= chunk_size:
            chunks.append({
                'content': para,
                'type': 'parent',
                'parent_id': parent_id,
                'child_id': None
            })
        else:
            # Split large paragraph into child chunks
            words = para.split()
            current_chunk = []
            child_id = 0
            
            for word in words:
                current_chunk.append(word)
                if len(' '.join(current_chunk)) >= chunk_size:
                    child_id += 1
                    chunks.append({
                        'content': ' '.join(current_chunk),
                        'type': 'child',
                        'parent_id': parent_id,
                        'child_id': child_id
                    })
                    current_chunk = []
            
            # Add remaining words
            if current_chunk:
                child_id += 1
                chunks.append({
                    'content': ' '.join(current_chunk),
                    'type': 'child',
                    'parent_id': parent_id,
                    'child_id': child_id
                })
    
    return chunks

# Test hierarchical chunking
long_document = """
Artificial intelligence has revolutionized many aspects of modern technology. Machine learning, a subset of AI, enables computers to learn from data without explicit programming. Deep learning, which uses neural networks with multiple layers, has achieved remarkable success in image recognition, natural language processing, and game playing.

Natural language processing (NLP) focuses on enabling computers to understand, interpret, and generate human language. Recent advances in transformer architectures, particularly models like BERT and GPT, have dramatically improved NLP capabilities. These models can perform tasks such as translation, summarization, question answering, and text generation with impressive accuracy.

Computer vision involves teaching machines to interpret and understand visual information from the world. Convolutional neural networks (CNNs) have been particularly effective for image classification, object detection, and image segmentation. Applications range from autonomous vehicles to medical diagnosis, facial recognition, and augmented reality.
"""

chunks = hierarchical_chunk_document(long_document, chunk_size=200)

print(f"\nCreated {len(chunks)} hierarchical chunks:\n")
for i, chunk in enumerate(chunks, 1):
    print(f"{i}. [{chunk['type'].upper()}] Parent: {chunk['parent_id']}, Child: {chunk['child_id']}")
    print(f"   {chunk['content'][:100]}...\n")


Created 6 hierarchical chunks:

1. [CHILD] Parent: 1, Child: 1
   Artificial intelligence has revolutionized many aspects of modern technology. Machine learning, a su...

2. [CHILD] Parent: 1, Child: 2
   neural networks with multiple layers, has achieved remarkable success in image recognition, natural ...

3. [CHILD] Parent: 2, Child: 1
   Natural language processing (NLP) focuses on enabling computers to understand, interpret, and genera...

4. [CHILD] Parent: 2, Child: 2
   have dramatically improved NLP capabilities. These models can perform tasks such as translation, sum...

5. [CHILD] Parent: 3, Child: 1
   Computer vision involves teaching machines to interpret and understand visual information from the w...

6. [CHILD] Parent: 3, Child: 2
   object detection, and image segmentation. Applications range from autonomous vehicles to medical dia...



In [14]:
def long_context_rag(query: str, long_doc: str) -> Dict[str, Any]:
    """
    RAG for long documents using hierarchical chunking.
    """
    print(f"\nProcessing long document RAG query: {query}\n")
    
    # Create hierarchical chunks
    chunks = hierarchical_chunk_document(long_doc, chunk_size=300)
    print(f"1️⃣  Created {len(chunks)} hierarchical chunks")
    
    # Convert to documents
    docs = [
        Document(
            content=chunk['content'],
            meta={
                'type': chunk['type'],
                'parent_id': chunk['parent_id'],
                'child_id': chunk['child_id']
            }
        )
        for chunk in chunks
    ]
    
    # Store and retrieve
    doc_store = InMemoryDocumentStore()
    doc_store.write_documents(docs)
    
    retriever = InMemoryBM25Retriever(document_store=doc_store)
    result = retriever.run(query=query, top_k=3)
    retrieved = result['documents']
    
    print(f"2️⃣  Retrieved {len(retrieved)} relevant chunks")
    
    # If child chunk retrieved, also get parent context
    expanded_docs = []
    for doc in retrieved:
        expanded_docs.append(doc)
        
        # If it's a child chunk, find and add parent
        if doc.meta['type'] == 'child':
            parent_id = doc.meta['parent_id']
            # Find parent or sibling chunks
            for other_doc in docs:
                if (other_doc.meta['parent_id'] == parent_id and 
                    other_doc.id != doc.id and
                    other_doc not in expanded_docs):
                    expanded_docs.append(other_doc)
    
    print(f"3️⃣  Expanded to {len(expanded_docs)} chunks (including parent context)")
    
    # Generate answer
    template = """
    Answer the question using the document excerpts below.
    
    Document excerpts:
    {% for doc in documents %}
    {{ doc.content }}
    {% endfor %}
    
    Question: {{ query }}
    
    Answer:
    """
    
    prompt_builder = PromptBuilder(template=template)
    prompt = prompt_builder.run(documents=expanded_docs, query=query)['prompt']
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    answer = llm.run(prompt=prompt)['replies'][0]
    
    return {
        'answer': answer,
        'chunks_retrieved': len(retrieved),
        'chunks_used': len(expanded_docs)
    }

# Test long-context RAG
result = long_context_rag(
    "What are transformer architectures used for?",
    long_document
)

print(f"\nAnswer: {result['answer']}")
print(f"\nChunks retrieved: {result['chunks_retrieved']}")
print(f"Chunks used (with parents): {result['chunks_used']}")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Processing long document RAG query: What are transformer architectures used for?

1️⃣  Created 6 hierarchical chunks
2️⃣  Retrieved 3 relevant chunks
3️⃣  Expanded to 6 chunks (including parent context)

Answer: Transformer architectures are used for natural language processing tasks, including translation, summarization, question answering, and text generation. Models like BERT and GPT, which are based on transformer architectures, have significantly advanced NLP capabilities.

Chunks retrieved: 3
Chunks used (with parents): 6


## 6. Multimodal RAG (Text + Images)

Process documents containing both text and images.

### 6.1 Conceptual Multimodal RAG

**Note:** Full multimodal RAG requires:
- Vision models (CLIP, GPT-4 Vision, etc.)
- Image embeddings
- Multimodal document stores

Here we'll demonstrate the concept with text descriptions of images.

In [15]:
# Simulate multimodal documents (in production, would use actual images + CLIP embeddings)
def create_multimodal_docs() -> List[Document]:
    """
    Create documents with text and image descriptions.
    In production, you'd extract actual visual features.
    """
    multimodal_data = [
        {
            "text": "Neural network architecture diagram showing input layer, hidden layers, and output layer.",
            "image_description": "Diagram with circles representing neurons connected by arrows showing forward propagation. Three distinct layers visible.",
            "image_type": "diagram",
            "has_image": True
        },
        {
            "text": "Convolutional neural networks apply filters to extract features from images.",
            "image_description": "Illustration showing convolution operation with a 3x3 filter sliding over an image matrix.",
            "image_type": "illustration",
            "has_image": True
        },
        {
            "text": "Transformers use self-attention mechanisms to process sequences.",
            "image_description": None,
            "image_type": None,
            "has_image": False
        },
        {
            "text": "Gradient descent optimization visualized on a loss surface.",
            "image_description": "3D surface plot showing a ball rolling down a curved surface toward the minimum point.",
            "image_type": "graph",
            "has_image": True
        }
    ]
    
    docs = []
    for data in multimodal_data:
        # Combine text and image description for retrieval
        if data['has_image']:
            combined_content = f"{data['text']}\n\nVisual content: {data['image_description']}"
        else:
            combined_content = data['text']
        
        docs.append(Document(
            content=combined_content,
            meta={
                'text': data['text'],
                'has_image': data['has_image'],
                'image_type': data['image_type']
            }
        ))
    
    return docs

multimodal_docs = create_multimodal_docs()
multimodal_store = InMemoryDocumentStore()
multimodal_store.write_documents(multimodal_docs)

print(f"✅ Created {len(multimodal_docs)} multimodal documents")
print(f"   {sum(1 for d in multimodal_docs if d.meta['has_image'])} with images")

✅ Created 4 multimodal documents
   3 with images


In [16]:
def multimodal_rag(query: str) -> Dict[str, Any]:
    """
    Simulated multimodal RAG.
    In production, would use vision models for actual image understanding.
    """
    print(f"\nMultimodal RAG Query: {query}\n")
    
    # Retrieve documents (text + image descriptions)
    retriever = InMemoryBM25Retriever(document_store=multimodal_store)
    result = retriever.run(query=query, top_k=3)
    docs = result['documents']
    
    print(f"Retrieved {len(docs)} multimodal documents:")
    for doc in docs:
        has_img = "📊 (with image)" if doc.meta['has_image'] else "📄 (text only)"
        print(f"  {has_img} {doc.meta['text'][:60]}...")
    
    # Generate answer incorporating visual information
    template = """
    Answer the question using both text and visual information from the documents.
    
    Documents:
    {% for doc in documents %}
    {{ doc.content }}
    ---
    {% endfor %}
    
    Question: {{ query }}
    
    Provide a comprehensive answer that references both textual and visual information when relevant.
    Answer:
    """
    
    prompt_builder = PromptBuilder(template=template)
    prompt = prompt_builder.run(documents=docs, query=query)['prompt']
    
    llm = OpenAIGenerator(api_key=Secret.from_env_var("OPENAI_API_KEY"), model="gpt-4o")
    answer = llm.run(prompt=prompt)['replies'][0]
    
    return {
        'answer': answer,
        'docs_with_images': sum(1 for d in docs if d.meta['has_image'])
    }

# Test multimodal RAG
multimodal_queries = [
    "How does a neural network process information?",
    "Show me how convolution works",
    "Explain gradient descent"
]

for query in multimodal_queries:
    result = multimodal_rag(query)
    print(f"\nAnswer: {result['answer']}")
    print(f"Documents with images: {result['docs_with_images']}/3\n")
    print("="*80)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Multimodal RAG Query: How does a neural network process information?

Retrieved 3 multimodal documents:
  📄 (text only) Transformers use self-attention mechanisms to process sequen...
  📊 (with image) Neural network architecture diagram showing input layer, hid...
  📊 (with image) Convolutional neural networks apply filters to extract featu...


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Answer: A neural network processes information by simulating the way biological brains process data, using interconnected layers of nodes (or neurons). The textual and visual information provided outlines different aspects of neural network architectures, specifically the general structure, self-attention mechanisms in transformers, and convolutional operations in convolutional neural networks (CNNs).

From the text, we know that transformers utilize self-attention mechanisms. This means they can weigh the significance of different elements within a sequence, allowing them to handle sequential data more effectively by considering the entire context of the input.

The visual information complements this by showing the general architecture of a neural network, comprising an input layer, hidden layers, and an output layer. These layers depict circles representing neurons connected by arrows, illustrating forward propagation. During forward propagation, inputs are received at the input la

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Answer: Convolutional neural networks (CNNs) utilize a process known as convolution to extract features from images. This is achieved through the application of filters, or kernels, that systematically scan over the image data to detect and extract specific patterns or features.

Textual Information: According to the document, convolutional neural networks apply filters to extract features from images, which involves using mathematical operations to highlight important features like edges, textures, or more complex patterns in the images.

Visual Information: The accompanying visual content demonstrates this process by illustrating a convolution operation with a 3x3 filter. In this depiction, the filter "slides" over the image matrix, performing mathematical operations (such as summing products) at each position to produce a new matrix, often referred to as a feature map. This matrix represents the presence of specific features detected by the filter.

Together, these components demon

### 6.2 Production Multimodal RAG

For production multimodal RAG:

```python
# Use CLIP for image embeddings
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Embed images
inputs = processor(images=image, return_tensors="pt")
image_embedding = model.get_image_features(**inputs)

# Or use GPT-4 Vision for image understanding
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4-vision-preview",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What's in this image?"},
            {"type": "image_url", "image_url": {"url": image_url}}
        ]
    }]
)
```

## 7. Comparison & Selection Guide

In [ ]:
# Create comparison table
comparison_data = [
    {
        'Technique': 'CRAG',
        'Best For': 'High-stakes domains (medical, legal)',
        'Complexity': 'Medium',
        'Latency': 'Higher (validation steps)',
        'Accuracy Gain': 'High (fewer errors)',
        'Setup Effort': 'Medium'
    },
    {
        'Technique': 'GraphRAG',
        'Best For': 'Complex relationships, multi-hop',
        'Complexity': 'High',
        'Latency': 'Medium',
        'Accuracy Gain': 'Very High (structured knowledge)',
        'Setup Effort': 'High (need graph DB)'
    },
    {
        'Technique': 'Long-Context',
        'Best For': 'Full document comprehension',
        'Complexity': 'Medium',
        'Latency': 'Medium',
        'Accuracy Gain': 'Medium (context preservation)',
        'Setup Effort': 'Low'
    },
    {
        'Technique': 'Multimodal',
        'Best For': 'Documents with images/diagrams',
        'Complexity': 'High',
        'Latency': 'Higher (vision processing)',
        'Accuracy Gain': 'High (visual understanding)',
        'Setup Effort': 'High (vision models)'
    }
]

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("SPECIALIZED RAG TECHNIQUES COMPARISON")
print("="*100 + "\n")
print(comparison_df.to_string(index=False))

## 8. Exercises & Challenges

### Exercise 1: Combine CRAG + GraphRAG

Build a system that uses GraphRAG for structured queries and CRAG for validation.

In [ ]:
# Your code here
# TODO: Implement hybrid CRAG + GraphRAG system

### Exercise 2: Implement Auto-Merging Retrieval

For long-context RAG, implement auto-merging of related chunks.

In [ ]:
# Your code here
# TODO: Implement auto-merging for consecutive chunks

### Challenge: Build Production GraphRAG

Set up Neo4j and implement real GraphRAG with Cypher queries.

In [ ]:
# Your code here
# TODO: Connect to Neo4j
# TODO: Implement entity extraction
# TODO: Build and query knowledge graph

## 9. Summary & Next Steps

### What You've Learned

✅ **Corrective RAG (CRAG)**
- Quality assessment of retrievals
- Correction strategies
- Answer validation

✅ **GraphRAG**
- Knowledge graph concepts
- Graph traversal for multi-hop reasoning
- Entity extraction and linking

✅ **Long-Context RAG**
- Hierarchical chunking
- Parent-child relationships
- Context preservation

✅ **Multimodal RAG**
- Combining text and images
- Visual information processing
- Multimodal retrieval concepts

### Key Takeaways

1. **Specialized ≠ Always Better**: Choose based on your use case
2. **CRAG for High Stakes**: When accuracy is critical
3. **GraphRAG for Relationships**: Structured knowledge wins
4. **Long-Context for Documents**: Preserve structure
5. **Multimodal for Rich Content**: Don't ignore images

### What's Next?

In **Notebook 5 - RAG Evaluation, Optimization & Production**, you'll learn:

- 📊 **RAGAS Framework** - Comprehensive RAG evaluation
- ⚡ **Performance Optimization** - Latency and cost reduction
- 🧪 **A/B Testing** - Compare RAG configurations
- 🚀 **Production Deployment** - Real-world considerations
- 📈 **Monitoring** - Track RAG system health

### Additional Resources

- [Corrective RAG Paper](https://arxiv.org/abs/2401.15884)
- [GraphRAG by Microsoft](https://microsoft.github.io/graphrag/)
- [Neo4j + LLMs Guide](https://neo4j.com/developer/graph-data-science/llms/)
- [CLIP Paper](https://arxiv.org/abs/2103.00020)

---

**Amazing work!** You've mastered cutting-edge RAG techniques. Continue to **Notebook 5** to make everything production-ready! 🚀